In [1]:
import numpy as np
import pandas as pd
import csv
import sys
import jieba
from gensim.models import word2vec
import math
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_squared_log_error
from gensim.test.utils import common_texts
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.test.utils import get_tmpfile

In [2]:
## read documents
data = pd.read_csv("mingjiang/detail_topic_new.csv",header=None)
data.columns=["Auto_Index","Author","Title","Max_layer","Read","Time"]

red_pocket_title=[u'下雨天宅家里？万元巨款你不准备要了吗！<长沙转发有礼>',
u'【周年店庆邀请函】丨点开有惊喜，转发有红包',
u'名匠心，设计情丨2017名匠杯设计大赛盛大启动',
u'【转发有惊喜】名匠心，设计情丨2017名匠杯设计大赛盛大启动',
u'年度特权日 装修特权价｜名匠2017年度【消费者特权日】盛大启幕',
u'眼要急！手要快！装修特权卡全城抢疯了',
u'开业特权 尊享钜惠 | 名匠装饰集团5家地市新店齐开',
u'国庆装修大动作！如此疯狂钜惠怎能错过？（前3000名转发有红包）',
u'抢定年度特权卡，装修钜惠更疯狂！（转发有红包）',
u'抢定年度特权卡，装修钜惠更疯狂！']

In [3]:
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean([(np.abs(y_true[i] - y_pred[i]) / max(y_true[i],0.1)) for i in range(len(y_true))])

def word_cut(text):
    return " ".join(jieba.cut(text))

def loss_output(data,f,d1=0,d2=0):
    y=np.array([math.log(int(i)) for i in list(data["Read"])])
    if d1==0 and d2>0: X=data[["Network_dim"+str(i) for i in range(d2)]]
    elif d1>0 and d2==0: X=data[["Title_dim"+str(i) for i in range(d1)]]
    elif d1>0 and d2>0: X=data[["Title_dim"+str(i) for i in range(d1)]+["Network_dim"+str(i) for i in range(d2)]]
    reg = LinearRegression().fit(X, y)
    y_pred=reg.predict(X)
    mse=mean_squared_error(y, y_pred)
    mape=mean_absolute_percentage_error(y, y_pred)
    r2=r2_score(y, y_pred)
    msle=mean_squared_log_error(y, y_pred)
    print("Prediction on Log of Read: ", file=f)
    print("Mean squared error: %.4f, Mean squared log error: %.4f " %(mse,msle),file=f)
    print("R^2 score: %.5f, Mean average percentage error: %.3f%% " %(r2,mape),file=f)
    
    y=np.array([int(i) for i in list(data["Max_layer"])])
    reg = LinearRegression().fit(X, y)
    y_pred=reg.predict(X)
    mse=mean_squared_error(y, y_pred)
    mape=mean_absolute_percentage_error(y, y_pred)
    r2=r2_score(y, y_pred)
    msle=mean_squared_log_error(y, y_pred)
    print("Prediction on Max Layers: ", file=f)
    print("Mean squared error: %.4f, Mean squared log error: %.4f " %(mse,msle),file=f)
    print("R^2 score: %.5f, Mean average percentage error: %.3f%% " %(r2,mape),file=f)
    
def title2vec_read(d,data):
    title2vec=pd.read_csv('Doc2Vec_Output/title2vec_%d.csv' %d,header=None,skiprows=1,delimiter=" ")
    title2vec.columns=["Auto_Index"]+["Title_dim"+str(i) for i in range(d)]
    title2vec["Author"]=data.Author
    title2vec["Title"]=data.Title
    title2vec["Time"]=data.Time
    title2vec["Read"]=data.Read
    title2vec["Max_layer"]=data.Max_layer
    title2vec[["Read"]] = title2vec[["Read"]].astype(int)
    title2vec=title2vec[title2vec.Read>1]
    return title2vec

In [11]:
# Cut words
data['cutted']=data.Title.apply(word_cut)
cut_list=[]
for line in data['cutted']:
    t=line.split(' ')
    while '' in t:
        t.remove('')
    cut_list.append(t)

cut_list_re = []
stop_words = open('stopwords.txt',encoding="utf-8")
stop_words_text = stop_words.read()
stop_words.close()
stop_words_list = stop_words_text.split('\n')

word_list=[]
for line in cut_list:
    t=[]
    for word in line:
        if word not in stop_words_list and word!=" " and len(word)!=1 and not word.isdigit() and word.isalpha():
            t.append(word)
            if word not in word_list: word_list.append(word)
    cut_list_re.append(t)

In [5]:
size_list=[5,10,15,20,25,30]
for size in size_list:
    documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(cut_list_re)]
    # model = Doc2Vec(documents, vector_size=5, window=2, min_count=1, workers=4)
    model = Doc2Vec(documents, vector_size=size, alpha=0.025, min_alpha=0.025, dm=0, min_count=0, epochs=10)
    # model.save_word2vec_format('word2vec_10.csv',doctag_vec=False, word_vec=True)
    model.save_word2vec_format('Doc2Vec_Output/title2vec_%d.csv' %size,doctag_vec=True, word_vec=False)

In [6]:
# model.save('my_model_10.doc2vec')
# model = Doc2Vec.load(fname)

In [7]:
# for epoch in range(10):
#     if epoch%2==0:
#         print("Now cal. epoch "+str(epoch))
#     model.train(documents, total_examples=model.corpus_count, epochs=model.epochs)
#     model.alpha -= 0.002  # decrease the learning rate
#     model.min_alpha = model.alpha  # fix the learning rate, no decay

In [11]:
# Evaluation for Doc2Vec model only
f=open("Doc2Vec_Output/title2vec.txt","a")
size_list=[5,10,15,20,25,30]
for d1 in size_list:
    print("Dimension_%d: " %d1,file=f)
    title2vec=title2vec_read(d1,data)
    loss_output(title2vec,f,d1)
f.close()

In [10]:
f=open("Doc2Vec_Output/Without_RP_combine.txt","a")
d1, d2=5, 5
title2vec=title2vec_read(d1,data)
title2vec=title2vec[~title2vec.Title.isin(red_pocket_title)]

# Common path
CP_author_id=pd.read_csv('mingjiang/CommonPath/node2vec_0_0.0.authorlist',header=None,delimiter=",")
CP_author_id.columns=["Author","node_id"]
CP_node2vec=pd.read_csv('node2vec-master/emb/node2vec_0_0.0.emb',header=None,skiprows=1,delimiter=" ")
CP_node2vec.columns=["node_id"]+["Network_dim"+str(i) for i in range(d2)]
CP_author_node2vec=pd.merge(CP_author_id, CP_node2vec,on='node_id')
CP_author_title2vec_node2vec=pd.merge(CP_author_node2vec, title2vec, on='Author')
CP_author_title2vec_node2vec.to_csv("Doc2Vec_Output/Without_RP_CP_author_title2vec_node2vec.csv")
print("Common Path Method: \n", file=f)
print("\nTitle Only: ", file=f)
loss_output(CP_author_title2vec_node2vec,f,d1,0)
print("\nNetwork Only: ", file=f)
loss_output(CP_author_title2vec_node2vec,f,0,d2)
print("\nCombined Model: ", file=f)
loss_output(CP_author_title2vec_node2vec,f,d1,d2)

# Full Path
d2=5
pair_id=pd.read_csv('mingjiang/CommonPath/fullpath.authorlist',header=None,delimiter=",")
pair_id.columns=["Author","Title","node_id"]
for d2 in [5,10]:
    node2vec=pd.read_csv('node2vec-master/emb/fullpath_%d.emb' %d2,header=None,skiprows=1,delimiter=" ")
    node2vec.columns=["node_id"]+["Network_dim"+str(i) for i in range(d2)]
    pair_node2vec=pd.merge(pair_id, node2vec,on='node_id')
    pair_title2vec_node2vec=pd.merge(pair_node2vec, title2vec, on=['Author','Title'])
    pair_title2vec_node2vec.to_csv("Doc2Vec_Output/Without_RP_pair_%d_title2vec_node2vec.csv" %d2)
    print("\n\n\nFull Path Method, Dim = %d: " %d2, file=f)
    print("\nTitle Only: ", file=f)
    loss_output(pair_title2vec_node2vec,f,d1,0)
    print("\nNetwork Only: ", file=f)
    loss_output(pair_title2vec_node2vec,f,0,d2)
    print("\nCombined Model: ", file=f)
    loss_output(pair_title2vec_node2vec,f,d1,d2)
    
f.close()